In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s up with you?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—if the course is still open, you can usually join now. The exact answer depends on the course’s enrollment rules and deadlines.

If you want, send me:
- the course name
- the platform or school
- the current date/deadline you’re seeing

and I can help you figure out whether late enrollment is possible and what to do next.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [9]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() # raise an error if the request was unsuccessful
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)      

1342

In [13]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [18]:
search_results = search(question)
[doc["answer"] for doc in search_results]

['Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'Summer 2025.',
 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.']

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [24]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [25]:
response.usage

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=34, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=368)

In [26]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0004035

In [27]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [28]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
) 

In [29]:
response.output_text

'Yes, you can still join. If you want a certificate, though, you need to submit your project while submissions are still being accepted.'

In [30]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [31]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [32]:
answer = rag(question)
print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.


In [66]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Please send:\n- the course name or link\n- the platform/school offering it\n- where you saw it listed\n\nIn the meantime, check for:\n- “Enroll,” “Join,” or “Register” buttons\n- an application deadline\n- prerequisites\n- whether it’s self-paced or starts on a fixed date\n\nIf you want, I can also help you draft a message to the instructor or support team asking if late enrollment is possible.'

In [67]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [68]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [69]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_zPtOsmGeMZg6dPWy1XvsMv4C', name='search', type='function_call', id='fc_09263e07fdbccc49006a303f71860481a0a0176dd9cf186691', namespace=None, status='completed')]

In [70]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [71]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_zPtOsmGeMZg6dPWy1XvsMv4C', name='search', type='function_call', id='fc_09263e07fdbccc49006a303f71860481a0a0176dd9cf186691', namespace=None, status='completed')]

In [72]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [47]:
print(response.output_text)

Yes — you can still join.

If you want a certificate, make sure you submit your project while the course is still accepting submissions.


In [50]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(651, 31)

In [51]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [73]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [74]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [76]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment access late registration"}


In [79]:
messages = [
    {"role": "user", "content": "I just discovered the coreuse. Can I join it?"}
]

In [80]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"coreuse join can I join it membership eligibility"}
iteration #2...
ASSISTANT:
Yes — you can still join.

If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [81]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [82]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model server course FAQ"}
function_call: search {"query":"Ollama run locally install macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, first install it from the official download page:

- **macOS**: download and install the `.pkg`
- **Windows**: download and install the `.msi`
- **Linux**: run

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

After installation, start a model locally by running:

```bash
ollama run llama3
```

That will download the model and open a local chat session.

If you want to use Ollama as a local server, you can check it with:

```bash
curl http://localhost:11434
```

If you’re using Python, install the client with:

```bash
pip install ollama
```

and then use it like this:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response['message']['content'])
```



'To run **Ollama locally**, first install it from the official download page:\n\n- **macOS**: download and install the `.pkg`\n- **Windows**: download and install the `.msi`\n- **Linux**: run\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\nAfter installation, start a model locally by running:\n\n```bash\nollama run llama3\n```\n\nThat will download the model and open a local chat session.\n\nIf you want to use Ollama as a local server, you can check it with:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you’re using Python, install the client with:\n\n```bash\npip install ollama\n```\n\nand then use it like this:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nIf you’re having trouble starting the server, you can also try:\n\n```bash\nollama serve\n```\n\nor, in notebook environments:\n\n```bash\nnohup ollama serve > nohup.ou

In [83]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course FAQ enrollment late registration join after start discovered course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

You don’t need a special confirmation to start learning or submit homework while the forms are open. If you want a certificate, make sure you submit your project while submissions are still being accepted.

If you want, I can also help with questions about homework, certificates, or how to get started.


'Yes, you can still join the course.\n\nYou don’t need a special confirmation to start learning or submit homework while the forms are open. If you want a certificate, make sure you submit your project while submissions are still being accepted.\n\nIf you want, I can also help with questions about homework, certificates, or how to get started.'

In [84]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late discovered course enroll can I join course FAQ"}
iteration #2...
function_call: search {"query":"certificate join course while form open project submit accepting submissions peer review live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete it with a live cohort.

If you just want to learn, you can also follow along without needing a registration confirmation.

Would you like me to also explain how certification and project submission work?


'Yes, you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete it with a live cohort.\n\nIf you just want to learn, you can also follow along without needing a registration confirmation.\n\nWould you like me to also explain how certification and project submission work?'

In [85]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening meaning"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers a pawn on c4 to try to **control the center** and tempt Black’s d-pawn away from the center.

A few quick points:
- It’s called a **gambit** because White is offering a pawn.
- It’s one of the **oldest and most popular** chess openings.
- Despite the name, it’s usually **not a real sacrifice**—White often gets the pawn back or gains better position.

There are two main types:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also show you the **basic idea behind the Queen’s Gambit** or a **simple move-by-move example**.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers a pawn on c4 to try to **control the center** and tempt Black’s d-pawn away from the center.\n\nA few quick points:\n- It’s called a **gambit** because White is offering a pawn.\n- It’s one of the **oldest and most popular** chess openings.\n- Despite the name, it’s usually **not a real sacrifice**—White often gets the pawn back or gains better position.\n\nThere are two main types:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also show you the **basic idea behind the Queen’s Gambit** or a **simple move-by-move example**.'

In [86]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"gambit chess queen's gambit opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so I can’t answer that from the course materials.

If you meant something course-related, feel free to rephrase with more context. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant something course-related, feel free to rephrase with more context. Are there other areas you want to explore?'

In [87]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [88]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [89]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [90]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [91]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [92]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [93]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [94]:
result.cost

CostInfo(input_cost=Decimal('0.00107775'), output_cost=Decimal('0.000936'), total_cost=Decimal('0.00201375'))

In [95]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run install local how do I run Olama locally"}', c

In [96]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [97]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='HI', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"HI"}', call_id='call_2WgGijwX7u2y5icaguMLT53b', name='search', ty